# NB04 — Raster Extraction (SoilGrids 250m)

**Objective**: Extract modeled soil property values at each validated station coordinate,
directly from the SoilGrids 250m rasters embedded in the country ZIP archives.
These values serve as fallback in NB05 when observed (WoSIS) data is absent.

**Source selected**: `SoilGrids_250m/` folder inside each ZIP.
- Layer used: `{property}_0-5cm_mean.tif` (surface representative, 0–5 cm depth)
- Rasters are read **in-memory** from the ZIP — no disk extraction required.
- One ZIP per country (the largest, most complete one, determined by NB01).

**Properties extracted** (13 SoilGrids layers → mapped to WoSIS property names):

| SoilGrids file prefix | WoSIS property | Raw unit       | Divisor | Output unit   |
|-----------------------|----------------|----------------|---------|---------------|
| bdod                  | BD             | cg/cm³ ×10     | 100     | g/cm³         |
| cec                   | CEC            | mmol(c)/kg ×10 | 10      | cmol(c)/kg    |
| cfvo                  | CF             | cm³/dm³ ×10    | 10      | %             |
| clay                  | clay           | g/kg ×10       | 10      | g/kg          |
| nitrogen              | N              | cg/kg ×100     | 100     | g/kg          |
| ocd                   | occ            | g/dm³ ×10      | 10      | g/dm³         |
| phh2o                 | pH             | pH ×10         | 10      | pH            |
| sand                  | sand           | g/kg ×10       | 10      | g/kg          |
| silt                  | silt           | g/kg ×10       | 10      | g/kg          |
| soc                   | occ (carbon)   | dg/kg          | 10      | g/kg          |
| wv0010                | WR_volumetric  | 0.001 cm³/cm³  | 1000    | cm³/cm³       |
| wv0033                | WR_volumetric  | 0.001 cm³/cm³  | 1000    | cm³/cm³       |
| wv1500                | WR_volumetric  | 0.001 cm³/cm³  | 1000    | cm³/cm³       |

**Inputs**:
- `data/intermediate/wosis_surface_qc.parquet` (from NB03)
- `data/intermediate/audit/zip_inventory.csv` (from NB01)
- `data/country/*` (ZIP archives)

**Outputs**:
- `data/intermediate/raster_cache/{iso3}.parquet` — per-country checkpoint
- `data/intermediate/soilgrids_surface_all.parquet` — merged all countries
- `logs/nb04_raster_extraction.log`

**Restart-safe**: if `raster_cache/{iso3}.parquet` already exists, that country is skipped.

**STOP condition**: if a country has 0% extraction success (all nodata) → logged as ERROR.

In [1]:
import io
import logging
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio
from rasterio.errors import NotGeoreferencedWarning
import warnings
warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
COUNTRY_DIR = BASE / 'data' / 'country'
INTER_DIR   = BASE / 'data' / 'intermediate'
CACHE_DIR   = INTER_DIR / 'raster_cache'
AUDIT_DIR   = INTER_DIR / 'audit'
LOG_DIR     = BASE / 'logs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb04_raster_extraction.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb04')
log.info(f'NB04 started at {datetime.now().isoformat()}')

# ── Usable QC flags (must match NB03) ─────────────────────────────────────
USABLE_FLAGS = {'VALID', 'NEAR_BORDER', 'OVERSEAS_TERRITORY'}

# ── SoilGrids property mapping ─────────────────────────────────────────────
# Format: sg_prefix → (wosis_property_name, divisor, output_unit)
# Divisors convert stored integer values to physically meaningful units.
# Source: ISRIC SoilGrids 250m v2.0 documentation.
SG_PROPERTIES = {
    'bdod':     ('BD',             100,  'g/cm3'),
    'cec':      ('CEC',             10,  'cmol(c)/kg'),
    'cfvo':     ('CF',              10,  'pct'),
    'clay':     ('clay',            10,  'g/kg'),
    'nitrogen': ('N',              100,  'g/kg'),
    'ocd':      ('occ',             10,  'g/dm3'),
    'phh2o':    ('pH',              10,  'pH'),
    'sand':     ('sand',            10,  'g/kg'),
    'silt':     ('silt',            10,  'g/kg'),
    'soc':      ('occ_carbon',      10,  'g/kg'),
    'wv0010':   ('WR_volumetric',  1000, 'cm3/cm3'),
    'wv0033':   ('WR_volumetric2', 1000, 'cm3/cm3'),
    'wv1500':   ('WR_gravimetric', 1000, 'cm3/cm3'),
}

# Surface layer depth label in SoilGrids filenames
SG_SURFACE_DEPTH = '0-5cm'

print(f'SoilGrids properties to extract: {len(SG_PROPERTIES)}')
for prefix, (wosis_name, divisor, unit) in SG_PROPERTIES.items():
    print(f'  {prefix:12s} → {wosis_name:18s}  ÷{divisor:<6}  [{unit}]')

2026-03-19 23:05:00,110 [INFO] NB04 started at 2026-03-19T23:05:00.110864


SoilGrids properties to extract: 13
  bdod         → BD                  ÷100     [g/cm3]
  cec          → CEC                 ÷10      [cmol(c)/kg]
  cfvo         → CF                  ÷10      [pct]
  clay         → clay                ÷10      [g/kg]
  nitrogen     → N                   ÷100     [g/kg]
  ocd          → occ                 ÷10      [g/dm3]
  phh2o        → pH                  ÷10      [pH]
  sand         → sand                ÷10      [g/kg]
  silt         → silt                ÷10      [g/kg]
  soc          → occ_carbon          ÷10      [g/kg]
  wv0010       → WR_volumetric       ÷1000    [cm3/cm3]
  wv0033       → WR_volumetric2      ÷1000    [cm3/cm3]
  wv1500       → WR_gravimetric      ÷1000    [cm3/cm3]


In [2]:
# ── Load inputs ────────────────────────────────────────────────────────────
# Station coordinates (QC-validated)
qc = pd.read_parquet(INTER_DIR / 'wosis_surface_qc.parquet')
stations = qc[qc['QC_FLAG'].isin(USABLE_FLAGS)][['station_id', 'lat', 'lon', 'iso3', 'country']].copy()
log.info(f'Stations to process: {len(stations):,} (after QC filter)')

# Inventory: best ZIP per country (largest file = most complete rasters)
inventory = pd.read_csv(AUDIT_DIR / 'zip_inventory.csv')
best_zip = (
    inventory[(inventory['has_csv'] == True) & (inventory['error'].isna())]
    .sort_values('zip_size_mb', ascending=False)
    .drop_duplicates('iso3')
    .set_index('iso3')[['zip_hash', 'zip_size_mb']]
)

countries = sorted(stations['iso3'].unique())
log.info(f'Countries to process: {len(countries)}')
print(f'Stations  : {len(stations):,}')
print(f'Countries : {len(countries)}')

# Check all countries have a ZIP
missing_zip = [c for c in countries if c not in best_zip.index]
if missing_zip:
    log.warning(f'No ZIP found for countries: {missing_zip}')
    print(f'WARNING: No ZIP for: {missing_zip}')

2026-03-19 23:05:00,169 [INFO] Stations to process: 48,435 (after QC filter)


2026-03-19 23:05:00,177 [INFO] Countries to process: 23


Stations  : 48,435
Countries : 23


In [3]:
# ── Core extraction function ───────────────────────────────────────────────

def extract_soilgrids_for_country(
    iso3: str,
    zip_path: Path,
    station_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Extract SoilGrids 0-5cm values at all station coordinates for one country.

    Strategy:
    - Read each raster from the ZIP into memory (no disk extraction).
    - Sample values using rasterio at (lon, lat) pairs.
    - Replace nodata (-9999) with NaN.
    - Apply unit conversion divisor.
    - Return one row per station with one column per SoilGrids property.
    """
    result = station_df[['station_id', 'lat', 'lon', 'iso3', 'country']].copy()
    coords = list(zip(station_df['lon'], station_df['lat']))  # rasterio expects (x, y) = (lon, lat)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zip_names = set(zf.namelist())

        for sg_prefix, (wosis_name, divisor, unit) in SG_PROPERTIES.items():
            col_name  = f'sg_{wosis_name}'      # output column: sg_BD, sg_clay, etc.
            tif_path  = f'SoilGrids_250m/{sg_prefix}_{SG_SURFACE_DEPTH}_mean.tif'

            if tif_path not in zip_names:
                log.warning(f'  {iso3}: {tif_path} not found — column set to NaN')
                result[col_name] = np.nan
                continue

            try:
                with zf.open(tif_path) as f:
                    raw_bytes = f.read()

                with rasterio.MemoryFile(raw_bytes) as memfile:
                    with memfile.open() as ds:
                        nodata = ds.nodata if ds.nodata is not None else -9999

                        # Clip coords to raster extent to avoid out-of-bounds errors
                        bounds = ds.bounds
                        clipped_coords = [
                            (max(bounds.left, min(lon, bounds.right)),
                             max(bounds.bottom, min(lat, bounds.top)))
                            for lon, lat in coords
                        ]

                        raw_values = np.array([v[0] for v in ds.sample(clipped_coords)], dtype=float)

                # Replace nodata with NaN, then apply unit conversion
                raw_values[raw_values == nodata] = np.nan
                result[col_name] = raw_values / divisor

            except Exception as e:
                log.error(f'  {iso3}: failed reading {tif_path}: {e}')
                result[col_name] = np.nan

    return result

In [4]:
# ── Per-country extraction loop (restart-safe via checkpoints) ─────────────
extraction_errors = []

for iso3 in countries:
    checkpoint = CACHE_DIR / f'{iso3}.parquet'

    # Skip if checkpoint already exists (restart-safe)
    if checkpoint.exists():
        log.info(f'  {iso3}: checkpoint exists — skipped')
        continue

    if iso3 not in best_zip.index:
        log.error(f'  {iso3}: no ZIP available — skipped')
        extraction_errors.append(iso3)
        continue

    zip_hash = best_zip.loc[iso3, 'zip_hash']
    zip_path = COUNTRY_DIR / str(zip_hash)
    country_stations = stations[stations['iso3'] == iso3].copy()

    try:
        result = extract_soilgrids_for_country(iso3, zip_path, country_stations)

        # Count valid (non-NaN) extractions per property
        sg_cols   = [c for c in result.columns if c.startswith('sg_')]
        n_stations = len(result)
        coverage  = {c: result[c].notna().sum() for c in sg_cols}
        best_cov  = max(coverage.values()) if coverage else 0
        pct_best  = 100 * best_cov / n_stations if n_stations > 0 else 0

        # Log per-country summary
        cov_str = ', '.join(f'{c.replace("sg_","")}:{v}' for c, v in coverage.items() if v > 0)
        log.info(f'  {iso3}: {n_stations} stations | best coverage {pct_best:.0f}% | {cov_str}')

        if best_cov == 0:
            log.error(f'  {iso3}: ALL properties returned nodata — investigate ZIP raster extent')
            extraction_errors.append(iso3)

        # Save checkpoint
        result.to_parquet(checkpoint, index=False)

    except Exception as e:
        log.error(f'  {iso3}: extraction failed — {e}')
        extraction_errors.append(iso3)

log.info(f'Extraction loop complete. Errors: {extraction_errors}')
print(f'\nExtraction complete. Errors: {extraction_errors if extraction_errors else "none"}')

2026-03-19 23:05:00,857 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:05:01,182 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:05:01,519 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:05:01,965 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:05:02,890 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


...

2026-03-19 23:06:36,679 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:06:36,831 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-19 23:06:36,886 [INFO]   TUN: 40 stations | best coverage 88% | BD:35, CEC:35, CF:35, clay:35, N:35, occ:35, pH:35, sand:35, silt:35, occ_carbon:35, WR_volumetric:35, WR_volumetric2:35, WR_gravimetric:35


2026-03-19 23:06:37,310 [INFO] Extraction loop complete. Errors: []



Extraction complete. Errors: none


In [5]:
# ── Merge all per-country checkpoints ─────────────────────────────────────
log.info('Merging per-country checkpoints...')

chunks = []
missing_checkpoints = []

for iso3 in countries:
    checkpoint = CACHE_DIR / f'{iso3}.parquet'
    if checkpoint.exists():
        chunks.append(pd.read_parquet(checkpoint))
    else:
        missing_checkpoints.append(iso3)
        log.warning(f'  {iso3}: no checkpoint found — excluded from merged output')

if not chunks:
    raise RuntimeError('STOP: No checkpoints found. Re-run the extraction loop.')

df_sg = pd.concat(chunks, ignore_index=True)
log.info(f'Merged: {len(df_sg):,} stations × {len(df_sg.columns)} columns')

if missing_checkpoints:
    print(f'WARNING: {len(missing_checkpoints)} countries missing: {missing_checkpoints}')

print(f'Merged SoilGrids table: {len(df_sg):,} stations × {len(df_sg.columns)} columns')

2026-03-19 23:06:38,002 [INFO] Merging per-country checkpoints...


2026-03-19 23:06:39,979 [INFO] Merged: 48,435 stations × 18 columns


Merged SoilGrids table: 48,435 stations × 18 columns


In [6]:
# ── Summary statistics ─────────────────────────────────────────────────────
sg_cols = [c for c in df_sg.columns if c.startswith('sg_')]
n_total = len(df_sg)

print('=== NB04 SUMMARY ===')
print(f'Total stations extracted : {n_total:,}')
print(f'Countries                : {df_sg["iso3"].nunique()}')
print(f'SoilGrids properties     : {len(sg_cols)}')
print()
print('SoilGrids coverage (% stations with non-NaN value):')
for col in sorted(sg_cols):
    pct = 100 * df_sg[col].notna().mean()
    bar = '#' * int(pct / 5)
    print(f'  {col:25s} {pct:5.1f}%  {bar}')

print()
print('Per-country extraction coverage (best property %):') 
cov = (
    df_sg.groupby('iso3')[sg_cols]
    .apply(lambda g: (g.notna().mean() * 100).max())
    .reset_index()
)
cov.columns = ['iso3', 'best_coverage_pct']
cov['best_coverage_pct'] = cov['best_coverage_pct'].round(1)
print(cov.sort_values('best_coverage_pct', ascending=False).to_string(index=False))

log.info('NB04 summary computed')

=== NB04 SUMMARY ===
Total stations extracted : 48,435
Countries                : 23
SoilGrids properties     : 13

SoilGrids coverage (% stations with non-NaN value):
  sg_BD                      97.4%  ###################
  sg_CEC                     97.4%  ###################
  sg_CF                      97.4%  ###################
  sg_N                       97.4%  ###################
  sg_WR_gravimetric          97.4%  ###################
  sg_WR_volumetric           97.4%  ###################
  sg_WR_volumetric2          97.4%  ###################
  sg_clay                    97.4%  ###################
  sg_occ                     97.4%  ###################
  sg_occ_carbon              97.4%  ###################
  sg_pH                      97.4%  ###################
  sg_sand                    97.4%  ###################
  sg_silt                    97.4%  ###################

Per-country extraction coverage (best property %):


2026-03-19 23:06:40,666 [INFO] NB04 summary computed


iso3  best_coverage_pct
 PNG              100.0
 DZA              100.0
 TCD              100.0
 MNE              100.0
 CAF              100.0
 BFA               99.9
 AGO               99.8
 EST               99.5
 CMR               98.9
 HRV               98.7
 BEN               98.3
 BWA               98.1
 AUS               97.6
 GRC               96.7
 BDI               96.4
 FRA               96.3
 DEU               94.8
 NLD               94.6
 GEO               94.4
 MDA               94.3
 MAR               93.9
 ALB               92.5
 TUN               87.5


In [7]:
# ── Save merged output ─────────────────────────────────────────────────────
out_path = INTER_DIR / 'soilgrids_surface_all.parquet'
df_sg.to_parquet(out_path, index=False)

log.info(f'Saved soilgrids_surface_all.parquet — {len(df_sg):,} rows')
log.info('NB04 completed successfully — proceed to NB05')

print(f'\nSaved: soilgrids_surface_all.parquet  {out_path.stat().st_size / 1e6:.2f} MB')
print(f'Per-country checkpoints: {CACHE_DIR}/')

2026-03-19 23:06:40,722 [INFO] Saved soilgrids_surface_all.parquet — 48,435 rows


2026-03-19 23:06:40,722 [INFO] NB04 completed successfully — proceed to NB05



Saved: soilgrids_surface_all.parquet  2.06 MB
Per-country checkpoints: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data/intermediate/raster_cache/
